# N001 · Notebook 与判题接口

**目标：** 独立运行单元格，并将普通函数包装为判题接口。

**先修：** 无。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** 内核与执行顺序；表达式与输出；Solution 与返回值；Restart and Run All。

**配套讲解来源：** [同名逐章意见](../../comment/001_jupyter_and_judge.md)。

本章阅读顺序：**先明白问题 → 补基础 → 手算 → 看核心片段 → 分段运行 → 核对测试**。需要复制使用时，跳到末尾“完整代码”，它包含全部依赖、实现和测试。

## 1. 通俗解释：输入什么，要得到什么

这一章是整个课程的第一课，它要解决的问题是：**你怎么在 Jupyter Notebook 里写一个函数，并且把它包装成 LeetCode 判题器能认识的样子**。

先看本章最终要交付的两个接口：`sum_two(a, b)` 和 `Solution.sum(num1, num2)`。前者是"自己的实现"，后者是"判题器认识的门面"，两者算的是同一件事。

在 LeetCode 上做题时，题目会给你一个固定的"壳"，比如 `class Solution` 里面有一个方法 `sum(self, num1, num2)`，你把代码填进去，判题器会调用这个方法，然后**读取它的返回值**来判断你对不对。注意，判题器看的不是你打印在屏幕上的东西，而是 `return` 出去的值。这就是"print 只是给人看的，return 才是给判题器看的"。

举个具体到数字的例子：输入 `a=2, b=3`，我们的函数 `sum_two(2, 3)` 会返回 `5`，因为 2 加 3 等于 5；如果输入是 `a=-4, b=4`，返回值是 `0`，因为 -4 加 4 正好抵消为 0。判题器拿到返回值 `5` 和 `0`，和标准答案一比，就知道你写对了。

这一章用最简单的"两个整数求和"当载体，真正要教的是三件事：notebook 的单元格要按顺序运行；表达式有输出值；函数要有返回值才能被判题器读到。

## 2. 基础知识：先读懂这些词

- **单元格（cell）**：notebook 里的代码是一小块一小块存的，每一块就是一个单元格。你可以单独运行任何一块，也可以从头到尾按顺序全部运行。
- **内核（kernel）**：内核就是那个真正在背后执行你 Python 代码的进程。你在单元格 A 里定义的变量和函数，都存活在内核的内存里；如果你只运行单元格 B 而没先运行 A，B 里用到的东西就不存在，会报 `NameError`。
- **执行顺序**：单元格编号 `In [3]`、`In [7]` 告诉你它们是第几个被运行的。你可以乱序运行（先跑第 5 格再跑第 2 格），这经常导致"看起来代码没问题却报错"，所以可靠的做法是 Restart and Run All。
- **Restart and Run All**：把内核彻底清空（内存里什么都没有了），然后从第一格到最后一格按顺序重跑一遍。如果全部能通过，说明你的 notebook 不依赖乱序运行的侥幸状态。
- **表达式与输出**：在单元格里写 `a + b` 这种表达式，notebook 会自动把它的值显示出来；但这个显示不等于函数返回值。`print` 是往屏幕上打字的"副作用"，`return` 才是把值交回给调用方的正式通道。
- **副作用（side effect）**：函数除了"算出返回值"之外对世界做的其他事情，比如打印、改文件。`print` 就是典型的副作用，判题器对它视而不见。
- **判题接口（Solution 壳）**：LeetCode 规定你必须把答案写进 `class Solution` 的指定方法里。本章的做法是：先写一个普通函数，再让 `Solution` 的方法转手调用它，这叫"委托"。
- **映射**：把程序看成一台"输入 → 输出"的机器：同样的输入永远得到同样的输出。求和函数就是最典型的映射，`sum_two(2, 3)` 永远是 `5`。
- **类型注解（type annotation）**：写在参数名后面冒号后的类型说明（如 `a: int`），以及写在 `->` 后面的返回类型说明。它是给人看的契约，Python 运行时不检查，但能帮你和读代码的人快速确认"这个函数吃什么、吐什么"。
- **委托（delegation）**：让一个函数（或方法）把真正的计算交给另一个函数完成，自己只负责传参数、传结果。本章 `Solution.sum` 委托给 `sum_two` 就是委托。
- **实例（instance）**：类的一次具体化。`Solution()` 这个表达式会现场造出一个 Solution 的实例，之后才能用点号调用它身上的方法：`Solution().sum(7,-2)`。

## 3. 思路推导：从小例子开始

我们的目标很小：算两个整数的和，并且以判题器认识的形式交出去。整体分五步。

- **Step 1：先写一个普通函数。** 我们定义 `sum_two(a, b)`，它只做一件事：把 `a + b` 算出来并 `return`。为什么不直接写在 `Solution` 里？因为普通函数方便我们单独测试、单独复用。
- **Step 2：再包一层判题壳。** 定义 `class Solution`，里面写方法 `sum(self, num1, num2)`，方法体只有一句 `return sum_two(num1, num2)`。这样判题器调用 `Solution().sum(...)` 时，实际干活的是我们自己的 `sum_two`。
- **Step 3：先手算，再跑代码验证。** 拿三个小例子（见第五节表格）：`(2, 3)`、`(-4, 4)`、`(0, 0)`，先在纸上算出 `5`、`0`、`0`，再让代码跑一遍，看两边是否一致。
- **Step 4：用断言固定结论。** 把"输入 → 期望输出"写成 `assert` 语句，只要实现改坏了，断言立刻报错。
- **Step 5：把整页从头到尾跑一遍。** 最后用 Restart and Run All 做一次"全链路彩排"，确认这套东西在干净内核上也能一路通过。

**手算演示一轮：** 判题器收到输入 `num1=7, num2=-2`。第一步，它调用 `Solution().sum(7, -2)`；第二步，`sum` 方法把 `7` 和 `-2` 原封不动交给 `sum_two`，也就是 `sum_two(7, -2)`；第三步，`sum_two` 计算 `7 + (-2) = 5` 并返回 `5`；第四步，这个 `5` 沿着调用链传回去，成为 `sum` 方法的返回值。判题器拿到 `5`，和标准答案一致，判定通过。注意整个过程没有任何 `print`，答案完全靠 `return` 这条通道传递。

**再手算一轮（sum_two(-4, 4)）：** 第一步读参数 a=-4、b=4；第二步做加法 `-4 + 4`，正负抵消得 `0`；第三步返回 `0`。对应到代码：这两次输入走的是**同一条代码路径**（同一个 `return a + b`），函数内部没有任何"记住上次结果"的东西，所以两个例子可以独立验证、独立通过。

**乱序运行的反面演示：** 假如你重启内核后先运行 “自动测试”部分 的测试格，它会立刻报 `NameError: name 'sum_two' is not defined`，因为定义 `sum_two` 的 “分段实现”部分 还没运行过。这就是"单元格可以乱序运行，但可靠的教程必须清空内核后按顺序运行"的直观含义。

**按顺序运行本章的检查单**（你也可以拿它当每一章的通用流程）：

1. 运行 “分段实现”部分，让内核记住 `sum_two` 和 `Solution` 的定义；
2. 运行 “运行示例”部分，让 `show_table` 先定义好再被调用，看到三行小实例表；
3. 运行 “自动测试”部分，三条断言依次通过，最后打印"所有本章断言通过"；
4. 中途如果改了 “分段实现”部分 的代码，要记得重跑 “运行示例”部分 和 “自动测试”部分——它们验证的是内核里**当前**这份定义，不是文件里那份。

## 4. 核心代码：sum_two

先关注决定答案的状态与更新操作。这是完整实现的对应片段，不是另一套解法；导入、辅助结构及其余接口在下一节补全。

```python
def sum_two(a: int, b: int) -> int:
    return a + b
```

### 逐段读懂代码

**核心实现（“分段实现”部分）对应实现：**

```python
def sum_two(a: int, b: int) -> int:
    return a + b

class Solution:

    def sum(self, num1: int, num2: int) -> int:
        return sum_two(num1, num2)
```

**第一段：`def sum_two(a: int, b: int) -> int:`**

这一行定义了一个名叫 `sum_two` 的函数，它接收两个参数 `a` 和 `b`。`a: int` 这种写法叫类型注解，意思是"我打算让 a 是个整数"，`-> int` 表示"这个函数打算返回一个整数"。注意：类型注解只是给人看的说明书，Python 并不会强制检查你真的传了整数。

**第二段：`return a + b`**

函数体只有一句话：把 `a` 和 `b` 相加，并把结果作为返回值交出去。`return` 一执行，函数就结束了。如果你把这一行写成 `print(a + b)`，屏幕上会显示结果，但函数的返回值会变成 `None`，判题器就会判错——这是本题最常见的坑。

**第三段：`class Solution:`**

定义一个类，名字必须是 `Solution`，这是 LeetCode 判题器指定的入口壳，名字写错判题器就找不到你的代码。

**第四段：`def sum(self, num1: int, num2: int) -> int:`**

在类里面定义方法 `sum`。它和普通函数的区别是第一个参数 `self`——Python 的实例方法约定第一个参数接收实例自身，它由 Python 自动传入，你调用时不用管它。方法名和调用接口应与题目模板一致。位置参数的局部名字可以改变；初学时保留模板命名更便于核对。

**第五段：`return sum_two(num1, num2)`**

方法体同样只有一句：把 `num1`、`num2` 转手交给前面定义的 `sum_two`，再把 `sum_two` 算出的结果直接返回。这就是"委托"——`Solution.sum` 自己不算，它只当个传话员。好处是：算法逻辑只存在一份（在 `sum_two` 里），我们平时测试普通函数，交题时套壳即可。

**小实例表格（“运行示例”部分）**：“运行示例”部分 定义了 `show_table` 辅助函数（用 HTML 渲染一张小表，不涉及算法），然后调用它展示三行数据——`输入 a=2, 输入 b=3, 返回值=5`；`输入 a=-4, 输入 b=4, 返回值=0`；`输入 a=0, 输入 b=0, 返回值=0`。这三行是当前内核现跑现算出来的，不是贴死的图片，所以它们同时验证了内核状态是正确的。逐行看这张表：第一行是"正数 + 正数"的正常情形；第二行是"负数 + 正数、和为零"的抵消情形；第三行是"零 + 零"的全零边界。三行加起来，正好把 “自动测试”部分 里前两条断言的期望值可视化了一遍——你先手算这三行，再跑 “运行示例”部分 对答案，最后跑 “自动测试”部分 让机器再确认一次，这就是"先手算，再运行"的完整闭环。

## 5. 分段实现：按顺序运行

**本章接口：** `sum_two(a, b)`、`Solution.sum(num1, num2)`

### sum_two

In [1]:
def sum_two(a: int, b: int) -> int:
    return a + b

### Solution

In [2]:
class Solution:

    def sum(self, num1: int, num2: int) -> int:
        return sum_two(num1, num2)

## 6. 运行示例与状态可视化

先预测结果再运行。手算表帮助解释过程；下面的表由当前实现计算。若表只展示输入输出，它并不代替内部状态轨迹。

In [3]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

show_table(['输入 a', '输入 b', '返回值'], [(a,b, sum_two(a,b)) for a,b in [(2,3),(-4,4),(0,0)]])

输入 a,输入 b,返回值
2,3,5
-4,4,0
0,0,0


## 7. 正确性、适用条件与复杂度

函数没有外部状态：每次调用只计算两个参数的和。用方法封装不改变该映射。

**代价：** 固定字长模型下时间、额外空间均为 O(1)；Python 大整数加法随位数增长。

### 展开理解

**正确性。** 题目要求返回两个整数的和。`sum_two(a, b)` 的返回表达式恰好是 `a + b`，因此直接满足目标。`Solution.sum` 原样传入两个参数并返回计算结果，所以这层接口不改变答案。没有外部状态有利于复现，但本身不是正确性证明：一个永远返回 0 的函数也没有外部状态，却不满足一般的求和要求。

**复杂度。** 固定字长模型下，一次加法需要 O(1) 时间和 O(1) 额外空间。Python 使用任意精度整数；按位数计成本时，两个至多 b 位的整数相加需要 O(b) 时间，结果也可能占 O(b) 空间。额外的函数调用会增加运行开销，但不改变这里的渐近量级，具体耗时不能脱离运行环境断言。

**边界。** `print(a+b)` 只展示信息，不能替代 `return a+b`。传入字符串时 `+` 会拼接而不是做整数加法；类型注解不会在运行时自动拒绝这类输入。正确性总是相对于已经约定的输入类型和目标而言。

## 8. 单元测试：每个用例在检查什么

“自动测试”部分 一共有三行断言，我们逐条看：

```python
assert sum_two(2, 3) == 5

assert sum_two(-4, 4) == 0

assert Solution().sum(7, -2) == 5

print('N001: 所有本章断言通过')
```

- **`assert sum_two(2,3) == 5`**：这是最普通的正常值测试。2 加 3 等于 5 是人人会算的，用它来确认函数在"正数 + 正数"这种最常见情况下不出错。
- **`assert sum_two(-4,4) == 0`**：这是特殊值测试——一个负数加一个正数，正好互相抵消。它测的是"负数进来会不会算错"（比如有人误写成 `a - b` 或 `abs(a) + abs(b)`，这条都会挂）。结果为 0 也是个很好的边界检查点。
- **`assert Solution().sum(7,-2) == 5`**：这条测的是"判题壳"这条路通不通。`Solution()` 现场创建一个实例，调用它的 `sum` 方法，期望它经由委托返回 `7 + (-2) = 5`。前两条证明普通函数是对的，这条证明包装层没有把值弄丢。注意它还顺带测了正负混合 `7 + (-2)`。

**如果实现写错了，哪条断言会先响？** 我们做个故障推演，帮你建立"断言 = 报警器"的直觉：

- 把 `return a + b` 写成 `return a - b`：第一条立刻挂（`2-3 = -1 ≠ 5`），第二条也会挂（`-4-4 = -8 ≠ 0`）；
- 把 `return` 写成 `print`：三条全挂，因为返回值全是 `None`，`None == 5` 不成立——这正是练习 1 要你体会的点；
- 委托时传错参数，比如 `return sum_two(num2, num1)`：本章这种交换对称的加法居然测不出来（结果还是 5），这提醒你：**测试通过不等于代码处处正确**，如果哪天算法换成减法，这条 bug 就会暴露。
- **`print('N001: 所有本章断言通过')`**：只有前面三条 assert 全部通过、程序没在中途报错退出时，这行才会被执行到。所以看到这句话打印出来，就代表三条测试全部通过。

另外注意原文的提醒：断言通过只说明"这几组输入对了"，不等于对所有可能的输入都证明了正确，也不等于已经在 LeetCode 官方提交过。

**把三条断言分个层，你会看到一套小而完整的测试观**：

| 断言 | 层次 | 它守护的契约 |
| --- | --- | --- |
| `sum_two(2,3) == 5` | 正常值 | 核心算术路径（正 + 正）是对的 |
| `sum_two(-4,4) == 0` | 特殊值 | 负数与抵消情形不崩、不错 |
| `Solution().sum(7,-2) == 5` | 接口层 | 判题壳的委托链路通了 |

以后每章的测试你都可以这样读：正常值管主干，特殊值管角落，接口层管"外面的人怎么用你"。本章只有三个输入维度可选（正、负、零），三条断言已经把普通函数和包装壳的两个入口都盖住了。

In [4]:
assert sum_two(2, 3) == 5

assert sum_two(-4, 4) == 0

assert Solution().sum(7, -2) == 5

print('N001: 所有本章断言通过')

N001: 所有本章断言通过


## 9. 练习：先尝试，再看提示

**练习 1：** 把打印答案改为返回答案。

**练习 2：** 清空内核后复现结果。

<details>
<summary>展开思路提示（不是完整答案）</summary>

- **练习 1（把打印答案改为返回答案）**：提示——先写一个"错误示范"：函数体里用 `print(a + b)` 而没有 `return`，然后调用它并用 `result = 函数名(...)` 接住返回值，你会发现 `result` 是 `None`，这就是判题器会判错的原因。改法很简单：把 `print` 换成 `return`。按题目要求，你要写清楚输入输出和边界（比如 `a=0, b=0` 时返回什么），先手算一个例子（如 `a=2, b=3`，期望返回 `5`），再用断言验证。这条练习的本质是让你亲手体会"print 给人看、return 给机器看"。验收时建议准备三个输入：正负混合（如 `7,-2`）、全零（`0,0`）、大一点的数（如 `100000, 1`），展示改前返回 `None`、改后返回正确值。
- **练习 2（清空内核后复现结果）**：提示——点击菜单里的 Restart Kernel（或 Restart and Run All），让内存清空，然后从本页第一格开始按顺序运行。注意观察：如果跳过定义 `sum_two` 的单元格直接跑测试格，会得到 `NameError`。这条练习验证的是你的 notebook 是"自包含"的——任何人拿到它，从头到尾跑一遍都能复现同样的表格和"所有本章断言通过"。验收时你还要能口头解释为什么正确，而不是只截图。具体可以按三步走：第一，Restart Kernel 后故意只跑 “自动测试”部分，记录报错信息；第二，按顺序跑 “分段实现”部分 → “运行示例”部分 → “自动测试”部分，记录三行表格和成功打印；第三，把两份记录放在一起，用一两句话解释"顺序为什么重要"。

</details>

## 10. 完整代码：可独立运行

下面这一格包含**全部导入、全部接口和验证用例**，不依赖前面的单元格。可以重启内核后只运行这一格，也可以复制到 `.py` 文件后运行。在 Notebook 中 Run All 时会再验证一次，这是为独立复制使用保留的完整版本。

In [5]:
# 完整实现：本单元格不依赖其他单元格。
def sum_two(a: int, b: int) -> int:
    return a + b

class Solution:

    def sum(self, num1: int, num2: int) -> int:
        return sum_two(num1, num2)

# 示例与回归测试
assert sum_two(2, 3) == 5

assert sum_two(-4, 4) == 0

assert Solution().sum(7, -2) == 5

print('N001: 所有本章断言通过')

N001: 所有本章断言通过


## 11. 代表题与迁移

- **2235. Add Two Integers**：这题就是本章的完整原型——给你两个整数 `num1` 和 `num2`，返回它们的和。它练的就是本章的核心点：把算法逻辑放进 `Solution` 方法的返回值里（对应 `Solution().sum(7,-2) == 5` 这条断言的完整流程）。做完这题你就完成了第一次"本地 notebook 写函数 → 包装成判题接口"的闭环。

**提交这题前的自检三问**：第一，我的方法是不是叫 `sum`、放在 `class Solution` 里？第二，我是用 `return` 交答案，而不是只 `print`？第三，参数顺序和题目给的 `num1`、`num2` 是不是对得上（加法虽然对称，但养成习惯）？三问都过，这题就是你的第一颗星。

另外提醒一句（对应 “题目映射”部分 原文的话）：本题列表只用于知识映射，LeetCode 的官方判题环境与本地 notebook 有差异（比如它是自己实例化 Solution 再喂测试数据），把"本地断言通过"理解成"逻辑大概率对"，提交后看官方反馈再修，是正常工作流。

**下一步**：接口和运行节奏都通了之后，你可以带着本章的三个问题进入 N002——"Python 里有哪些值和类型？运算符优先级谁先谁后？"；学 N002 时同样先看推导格、猜输出，再运行核对，把这套节奏固定下来。

**题面入口：** [2235](https://leetcode.com/problems/add-two-integers/)

题目用于知识映射；本章完整代码遵循教学接口，不等于每道题的完整平台提交。

## 12. 掌握检查

合上代码，说明输入输出、核心变量的含义、每次更新为什么成立，以及一个边界情况。再完成一道练习，不能只凭“运行通过”判断掌握。

**继续学习：** [课程索引](../../COURSE_INDEX.md)。